# Лабораторная работа – 7.
# Предобработка текста для решения задач NLP


Импорт библиотек

In [40]:
import pandas as pd
import re
import os
import nltk
from nltk.corpus import stopwords
from pymystem3 import Mystem
from sklearn.feature_extraction.text import TfidfVectorizer


Читаю файл и выбираю свой варинт

In [41]:
df = pd.read_csv("dialogues.tsv", sep="\t", on_bad_lines="skip", engine="python")

# 25-й вариант (индекс 24)
raw_data = df.loc[24, "dialogue"]

raw_data

'<span class=participant_2>Пользователь 2: Привет</span><br /><span class=participant_1>Пользователь 1: Привет</span><br /><span class=participant_2>Пользователь 2: Как тебя зовут? Чем занимаешься?</span><br /><span class=participant_1>Пользователь 1: Сколько тебе лет?????</span><br /><span class=participant_2>Пользователь 2: Мне 30, а тебе?</span><br /><span class=participant_1>Пользователь 1: Алевтина Егорова, а тебя??</span><br /><span class=participant_1>Пользователь 1: Пеку пирожки для своих внуков</span><br /><span class=participant_2>Пользователь 2: Дима, приятно познакомиться)</span><br /><span class=participant_1>Пользователь 1: Очень приятно))</span><br /><span class=participant_2>Пользователь 2: Я как раз учусь печь пироги)</span><br /><span class=participant_2>Пользователь 2: А профессия у тебя есть какая-нибудь?</span><br /><span class=participant_1>Пользователь 1: Моё хобби читать, а твоё??</span><br /><span class=participant_1>Пользователь 1: Нет, я пенсионерка</span><br

Создаю паттерн для поиска:

([12]) - 1 или 2 (у нас только 2 участника) и сохраняет цифру  в результат (в matches).

.*? - пропусти любые символы, пока не встретишь слово "Пользователь"

\d - любая цифра от 0 до 9

(.*?) - любые символы, сколько угодно раз, захвати всё до span, сохрани в отдельную переменную()

In [42]:
# Здесь по сути тоже удаляються HTML теги
pattern = r"<span class=participant_([12])>.*?Пользователь \d: (.*?)</span>"
matches = re.findall(pattern, raw_data)

print("Без объединения последовательных реплик одного участника")
matches

Без объединения последовательных реплик одного участника


[('2', 'Привет'),
 ('1', 'Привет'),
 ('2', 'Как тебя зовут? Чем занимаешься?'),
 ('1', 'Сколько тебе лет?????'),
 ('2', 'Мне 30, а тебе?'),
 ('1', 'Алевтина Егорова, а тебя??'),
 ('1', 'Пеку пирожки для своих внуков'),
 ('2', 'Дима, приятно познакомиться)'),
 ('1', 'Очень приятно))'),
 ('2', 'Я как раз учусь печь пироги)'),
 ('2', 'А профессия у тебя есть какая-нибудь?'),
 ('1', 'Моё хобби читать, а твоё??'),
 ('1', 'Нет, я пенсионерка'),
 ('2', 'Я не умею плавать, в ближайшее время хочу научиться'),
 ('2', 'А работаю я грузчиком'),
 ('1', 'А у меня трое внуков'),
 ('2', 'Здорово) у меня нет детей'),
 ('2', 'Только кактус'),
 ('1', 'И летом мы ходим в лес за грибами и ягодами')]

# Восстановление структуры диалога

In [43]:
dialogue_structured = []
current_user = None
current_text = ""

for user_id, text in matches:
    # Удаление HTML-тегов <br- начало тэга, \s* - любое кол-во пробелов,  />- конец тэга
    clean_text = re.sub(r"<br\s*/>", " ", text)
    # удаляет двойные/тройные пробелы внутри реплик и в начале и конце текста
    clean_text = re.sub(r"\s+", " ", clean_text).strip()

    if user_id == current_user:
        current_text += " " + clean_text
    else:
        if current_user is not None:
            dialogue_structured.append((current_user, current_text))
        current_user = user_id
        current_text = clean_text
if current_user is not None:
    dialogue_structured.append((current_user, current_text))

dialogue_structured

[('2', 'Привет'),
 ('1', 'Привет'),
 ('2', 'Как тебя зовут? Чем занимаешься?'),
 ('1', 'Сколько тебе лет?????'),
 ('2', 'Мне 30, а тебе?'),
 ('1', 'Алевтина Егорова, а тебя?? Пеку пирожки для своих внуков'),
 ('2', 'Дима, приятно познакомиться)'),
 ('1', 'Очень приятно))'),
 ('2', 'Я как раз учусь печь пироги) А профессия у тебя есть какая-нибудь?'),
 ('1', 'Моё хобби читать, а твоё?? Нет, я пенсионерка'),
 ('2',
  'Я не умею плавать, в ближайшее время хочу научиться А работаю я грузчиком'),
 ('1', 'А у меня трое внуков'),
 ('2', 'Здорово) у меня нет детей Только кактус'),
 ('1', 'И летом мы ходим в лес за грибами и ягодами')]

# Приведение текста к нормализованному виду
(избавляемся от разных форм слов ведь нам важен только смысл)

Лемматизатор приводит слова в словарную форму: пеку - печь

Работает дольше за счёт того, что смотрит в словарь, но подходит лучше на малом объёме данных и для русского языка 

Стемминг - это «грубое» обрезание окончаний он не знает языка, он просто отпиливает всё, что похоже на окончание. «корова» -> «коров»

Работает быстро но не хорошо подходит под русский язык

In [44]:
# объект лемматизатора
mystem = Mystem()
# Загружаю список мусорных слов (союзы, предлоги, частицы), которые не несут смысла для анализа
stop_words = set(stopwords.words("russian"))

def preprocess_text(text):
    text = text.lower()
    # всё, что НЕ является русскими буквами или пробелами заменяется на пробел
    text = re.sub(r"[^а-яё\s]", " ", text)
    # разбивает текст на слова и превращает каждое в начальную форм
    lemmas = mystem.lemmatize(text)
    # удаление стоп-слов (word not in stop_words) и пустых строк (word.strip())
    return " ".join(
        [word for word in lemmas if word.strip() and word not in stop_words]
    )

processed_corpus = [preprocess_text(text) for user, text in dialogue_structured]
print(processed_corpus[::1])

empty_count = processed_corpus.count("")
print(f"Всего реплик: {len(processed_corpus)}")
print(f"Пустых после очистки: {empty_count}")

['привет', 'привет', 'звать заниматься', 'сколько год', '', 'алевтина егорова печь пирожок свой внук', 'дима приятно познакомиться', 'очень приятно', 'учиться печь пирог профессия', 'хобби читать твой пенсионерка', 'уметь плавать близкий время хотеть научаться работать грузчик', 'трое внук', 'здорово ребенок кактус', 'лето ходить лес гриб ягода']
Всего реплик: 14
Пустых после очистки: 1


# Представление текстовых данных в числовом формате.

TfidfVectorizer
1. Составляет словарь уникальных слов
2. Cчитает вес каждого слова по формуле 
$$TF(t, d) = \frac{\text{Количество вхождений термина } t \text{ в документ } d}{\text{Общее количество терминов в документе } d}$$
$$IDF(t) = \log\left(\frac{N}{\text{Количество документов, содержащих термин } t}\right)$$
$$Вес = TF(t, d) \times IDF(t)$$
$$N — \text{общее количество всех документов}$$
3. Каждую реплику превращает в вектор (список чисел) в моём случае длина вектора 39
4. Если в реплике встретилось слово из словаря, то он ставит туда его вес, иначе 0

In [45]:
# vectorizer = TfidfVectorizer(ngram_range=(1, 2)) уже для n-грамм
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(processed_corpus)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns= vectorizer.get_feature_names_out()
)

print("Строки - реплики, столбцы - уникальные слова")
df_tfidf

Строки - реплики, столбцы - уникальные слова


,алевтина,близкий,внук,время,год,гриб,грузчик,дима,егорова,заниматься,...,сколько,твой,трое,уметь,учиться,хобби,ходить,хотеть,читать,ягода
0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
1,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
2,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.707107,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
3,0.00000,0.000000,0.000000,0.000000,0.707107,0.000000,0.000000,0.00000,0.00000,0.000000,...,0.707107,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
4,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
5,0.42647,0.000000,0.369116,0.000000,0.000000,0.000000,0.000000,0.00000,0.42647,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
6,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.60312,0.00000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
7,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
8,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.516459,0.0,0.000000,0.000000,0.0,0.000000
9,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,...,0.000000,0.5,0.000000,0.000000,0.000000,0.5,0.000000,0.000000,0.5,0.000000


Cкладываем веса каждого слова по всем репликам(если смотреть по таблице, то сверху вниз),  получаем суммарный вес слова для всего диалога целиком. A1 превращает результат из np массива в обычный

In [46]:
feature_names = vectorizer.get_feature_names_out()
word_scores = tfidf_matrix.sum(axis=0).A1

word_scores

array([0.42647023, 0.35355339, 1.02354782, 0.35355339, 0.70710678,
       0.4472136 , 0.35355339, 0.60311998, 0.42647023, 0.70710678,
       0.70710678, 0.57735027, 0.57735027, 0.4472136 , 0.4472136 ,
       0.35355339, 0.75612063, 0.5       , 0.81611744, 0.51645887,
       0.42647023, 0.35355339, 0.60311998, 2.        , 1.17644049,
       0.51645887, 0.35355339, 0.57735027, 0.42647023, 0.70710678,
       0.5       , 0.75612063, 0.35355339, 0.51645887, 0.5       ,
       0.4472136 , 0.35355339, 0.5       , 0.4472136 ])

# Вывести топ 10 важных слов из диалога

In [47]:
df_tfidf = pd.DataFrame({"word": feature_names, "score": word_scores})
top_10_words = df_tfidf.sort_values(by="score", ascending=False).head(10)

print("ТОП 10 важных слов в диалоге")
print(top_10_words.to_string(index=False))

ТОП 10 важных слов в диалоге
      word    score
    привет 2.000000
   приятно 1.176440
      внук 1.023548
      печь 0.816117
     очень 0.756121
      трое 0.756121
       год 0.707107
     звать 0.707107
   сколько 0.707107
заниматься 0.707107
